# Gold Layer: Equipment Anomaly Summary

**Purpose:** Identify equipment with data quality issues and sensor problems

**Why this table?**
- Quarantine table has 1,186 bad readings scattered across many equipment 
- Hard to see: "Whichequipment has the MOST problems?"
- This table answers: "Rank equipment by sensor health"

**Input:** `dev.bronze.sensor_readings_quarantine`
(The bad data we separated during Bronze ingestion)

**Output:** `dev.gold.equipment_anomaly_summary`

**Example Output:**
| Equipment ID | Equipment Type | Total Errors | Error Rate | Status | Action |
| ------------ | -------------- | ------------ | ---------- | ------ | ------ |
| EQ-0042      | CONVEYOR_BELT  | 145          | 12.05%     | CRITICAL | Replace Sensor |
| EQ-0015      | ROBOTIC_ARM    | 87           | 7.2%       | HIGH     | Inspect        |
| EQ-0099      | CNC_MACHINE    | 23           | 2.1%       | NORMAL   | Monitor        |

**Business Value:**
- Identifies failing sensors (high error rate)
- Prioritizes maintenance (which equipment first?)
- Tracks data quality trends (is it getting worse?)
- Guides sensor replacement decisions

## Configuration

In [0]:
# Configuration: Define input and ouput tables

# INPUT: The quarantine table from Bronze layer
# This contains all the bad readings we separated during ingestion
INPUT_QUARANTINE = "dev.bronze.sensor_readings_quarantine"

# INPUT: Equipment metadata (to get equipment details)
INPUT_EQUIPMENT = "dev.bronze.equipment_metadata_raw"

# INPUT: Good readings count (to calculate error rate)
INPUT_GOOD_READINGS = "dev.bronze.sensor_readings_raw"

# OUTPUT: Equipment anomaly summary
OUTPUT_TABLE = "dev.gold.equipment_anomaly_summary"

print("=" * 70)
print("CONFIGURATION: Equipment Anomaly Summary")
print("=" * 70)
print(f"Input (Quarantine): {INPUT_QUARANTINE}")
print(f"Input (Equipment): {INPUT_EQUIPMENT}")
print(f"Input (Good data): {INPUT_GOOD_READINGS}")
print(f"Output: {OUTPUT_TABLE}")
print(f"\nThis table identifies equipment with sensor problems")

## Load Data

Load quarantine data and equipment metadata

In [0]:
#  Load all threee input tables

# 1. Load QUARANTINE data (bad readings)
# Why? Count how many bad readings per equipment
quarantine_df = spark.read.table(INPUT_QUARANTINE)

# 2. Load EQUIPMENT metadata
# Why? Get equipment details (type, factory, criticality)
equipment_df = spark.read.table(INPUT_EQUIPMENT)

# 3. Load Good readings
# Why? Count total readings to calculate error percentage
good_df = spark.read.table(INPUT_GOOD_READINGS)

print(f"Data loaded:")
print(f"Quarantine records: {quarantine_df.count()}")
print(f"Equipment records: {equipment_df.count()}")
print(f"Good readings: {good_df.count()}")

print(f"Quarantine data sample:")
quarantine_df.select(
    "equipment_id",
    "sensor_type",
    "quality_flag",
    "value"
).show(5, truncate=False)

## Count Anomalies by equipment 

Count how many bad readings each equipment has

In [0]:
from pyspark.sql.functions import count, col, round as spark_round

# Coutn bad readings per equipment
# Why count? Shows which equipment has the most problems

quarantine_by_equipment = quarantine_df.groupBy(
    "equipment_id",
    "quality_flag"  # MISSING, INVALID, OUTLINER, etc.
).count().withColumnRenamed("count", "error_count")

print(f"Errors grouped by equipment and type")
print(f"\nErrors by equipment ID:")
quarantine_by_equipment.show(15, truncate=False)

# Total errors per equipment (sum across all error types)
# Why total? Get overall error count
total_errors_per_equipment = quarantine_df.groupBy(
    "equipment_id"
).count().withColumnRenamed("count", 'total_error_count')

print(f"\ntotal errors per equipment:")
total_errors_per_equipment.orderBy("total_error_count", ascending=False).show(20)

## Count Good Readings by Equipment

Get total readings per equipment (to calculate error rate)

In [0]:
# Count GOOD readings per equipment
# Why? Need this to calculate: error_rate = errors / (errors + good_readings)

good_readings_per_equipment = good_df.groupBy(
    "equipment_id"
).count().withColumnRenamed("count", "good_reading_count")

print(f"Good readings counted per equipment")
print(f"\nGood readings per equipment")
good_readings_per_equipment.orderBy("good_reading_count", ascending=False).show(10)

## Calculate Error Rate

Error Rate = Bad Readings / (Bad + Good) Readings * 100

In [0]:
# Combine errors and good readings, then calculate error percentage
# Why error rate? Show PROPORTION of bad data
# 100 errors out of 10,000 readings = 1% (not too bad)
# 100 errors out of 200 readings = 50% (very bad!)

error_rate_calculated = total_errors_per_equipment.join(
    good_readings_per_equipment,
    on="equipment_id",
    how="left"
).fillna(0) # If no good readings, fill with 0

# Calculate error rate
error_rate_calculated = error_rate_calculated.withColumn(
    "error_rate_percent",

    # error_rate = (errors / (errors + good)) * 100
    # Why * 100? Convert to percentage (0-100) instead of 0-1
    spark_round(
        (col("total_error_count") /
         (col("total_error_count") + col("good_reading_count"))) * 100,
        2
    )
)

print(f"Error rate calculated")
print(f"\nEquipment with error rates (sorted by rate)")
error_rate_calculated.orderBy("error_rate_percent", ascending=False).show(20, truncate=False)

## Join with Equipment Details

Add equipment metadata (type, factory, criticality)

In [0]:
# Select equipment columns we need
equipment_details = equipment_df.select(
    col("equipment_id"),
    col("equipment_name"),
    col("equipment_type"),
    col("manufacturer"),
    col("factory_location"),
    col("criticality")
)

# Join error rates with equipment details
# Why join? Get context: Is this critical equipment? Which factory?
anomaly_with_details = error_rate_calculated.join(
    equipment_details,
    on="equipment_id",
    how="left"
)

print(f"Equipment details joined")
print(f"\nAnomalies with equipment context:")
anomaly_with_details.select(
    "equipment_id",
    "equipment_type",
    "total_error_count",
    "error_rate_percent",
    "factory_location",
    "criticality"
).orderBy("error_rate_percent", ascending=False).show(10, truncate=False)

## Classify Anomaly Severity

Assign severity level based on error rate

In [0]:
from pyspark.sql.functions import when

# Classify severity based on error rate
# Why classify? Makes it eady to understand which equipment needs urgent attention

# ERROR RATE THRESHOLDS
# > 10% = CRITICAL (sensor definitely faling)
# > 5% = HIGH (sensor having issues)
# > 2% = MEDIUM (some issue, monitor)
# ≤ 2% = NORMAL (acceptable error rate)

severity_classified = anomaly_with_details.withColumn(
    "severity_level",

    # Classify based on error rate percentage
    when(col("error_rate_percent") > 10, "CRITICAL")
    .when(col("error_rate_percent") > 5, "HIGH")
    .when(col("error_rate_percent") > 2, "MEDIUM")
    .otherwise("NORMAL")
)

print(f"\nSample with severity:")
print("\nEquipment by severity:")
severity_classified.groupBy("severity_level").count().show()

print(f"\nSample with severity:")
severity_classified.select(
    "equipment_id",
    "equipment_type",
    "error_rate_percent",
    "severity_level"
).orderBy("error_rate_percent", ascending=False).show(15, truncate=False)

## Determine Recommended Action

What should maintenance do about each equipment?

In [0]:
# Recommend action based on severity
# Why actions? Help plant managers decide what to do

action_recommended = severity_classified.withColumn(
    "recommended_action",

    # CRITICAL: Replace sendor immediately
    when(col("severity_level") == "CRITICAL",
        "REPLACE_SENSOR - High failure rate detected"     
    )

    # HIGH: Inspect and potentially recalibrate
    .when(col("severity_level") == "HIGH",
        "INSPECT_SENSOR - Possible calibration drift"      
    )

    # MEDIUM: Monitor closely, plan replacement
    .when(col("severity_level") == "MEDIUM",
          "MONITOR - Schdule sensor replacement"
    )

    # NORMAL: Acceptable, continue normal monitorning
    .otherwise("NORMAL_OPERATION - Continue monitorring")
)

print(f"Recommended action assigned")
print(f"\nActions by severity")
action_recommended.groupBy("severity_level" , "recommended_action").count().show(10,truncate=False)

print(f"\nEquipment needing action:")
action_recommended.filter(col("severity_level") != "NORMAL").select(
    "equipment_id",
    "equipment_type",
    "error_rate_percent",
    "severity_level",
    "recommended_action"
).orderBy("error_rate_percent", ascending=False).show(15, truncate=False)

## Identify Error Patterns

What types of errors are most common per equipment?

In [0]:
# Analyze error types per equipment
# Why? Different error types indicate different problems:
# - MISSING = sensor not sending data (battery? connection?)
# - INVALID = wrong data type (sensor malfunction?)
# - OUTLIER = extreme values (sensor drift? interference?)

error_type_breakdown = quarantine_df.groupBy(
    "equipment_id",
    "quality_flag"
).count().withColumnRenamed("count", "count_by_type")

print(f"Error types analyzed")
print(f"\nError types by equipment (top 20)")
error_type_breakdown.orderBy("count_by_type", ascending=False).show(20, truncate=False)

# Which error types are most common overall?
print(f"\nMost common error types across all equipment:")
quarantine_df.groupBy("quality_flag").count().orderBy("count", ascending=False).show()

## Add Metadata and Timestamp

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Add when this anomaly summary was created
final_summary = action_recommended.withColumn(
    "summary_timestamp",
    current_timestamp()
).withColumn(
    "summary_source",
    lit("bronze_quarantine_table")  # Data source
)

print(f"Metadata added")

## Reorder Columns for Business Users

In [0]:
# Put most important columns first for business users

final_ouput = final_summary.select(
    # Equipment identification
    "equipment_id",
    "equipment_name",
    "equipment_type",
    "factory_location",

    # Error metrics
    "total_error_count",
    "good_reading_count",
    "error_rate_percent",

    # Health assessment
    "severity_level",
    "criticality",
    "recommended_action",

    # Metadata
    "manufacturer",
    "summary_timestamp"
)

print(f"Columns reordered")
print(f"Final anomaly summary:")
final_ouput.show(20, truncate=False)

## Write to Gold Table

In [0]:
# Write equipment anomal summary to Gold layer

print(f"Writing {final_ouput.count()} equipment anomal records...")

final_ouput.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(OUTPUT_TABLE)

print(f"Gold layer table written!")
print(f"Table: {OUTPUT_TABLE}")
print(f"Records: {final_ouput.count()}")

## Verify and Display Results

In [0]:
# Read back and verify the anomal summary

anomaly_verify = spark.read.table(OUTPUT_TABLE)

print("=" * 90)
print("EQUIPMENT ANOMALY SUMMARY (GOLD LAYER)")
print("=" * 90)

print(f"\nTotal equipment in summary: {anomaly_verify.count()}")

print(f"\nFull Anomaly Summary (sorted by error rate):")
anomaly_verify.orderBy("error_rate_percent", ascending=False).show(30, truncate=False)

## Business Insights and Analysis

In [0]:
from pyspark.sql.functions import avg as spark_avg

print("=" * 90)
print("EQUIPMENT ANOMALY INSIGHTS FOR PLANT MANAGERS")
print("=" * 90)

# How many equipment in each severtiy category?
print(f"EQUIPMENT BY SEVERITY LEVEL:")
anomaly_verify.groupBy("severity_level").count().show()

# Which equipment Need Action?
print(f"/EQUIPMENT REQUIRING IMMEDIATE ACTION (CRITICAL/HIGH):")
critical_high = anomaly_verify.filter(
    (col("severity_level") == "CRITICAL") | (col("severity_level") == "HIGH")
).orderBy("error_rate_percent", ascending=False)

critical_high.select(
    "equipment_id",
    "equipment_type",
    "error_rate_percent",
    "severity_level",
    "recommended_action"
).show(15, truncate=False)

print(f"\nTotal equipment needing action: {critical_high.count()}")

# Average error rate by factory
print(f"\n ERROR RATES BY FACTORY")
anomaly_verify.groupBy("factory_location").agg(
    spark_round(spark_avg("error_rate_percent"), 2).alias("avg_error_rate"),
    count("*").alias("equipment_count")
).show()

# Average error rate by equipment type
print("\n ERROR RATESBY EQUIPMENT TYPE:")
anomaly_verify.groupBy("equipment_type").agg(
    spark_round(spark_avg("error_rate_percent"), 2).alias("avg_error_rate"),
    count("*").alias("equipment_count")
).orderBy("avg_error_rate", ascending=False).show()

# Which sensors fail most oftern?
anomaly_verify.orderBy("error_rate_percent", ascending=False).select(
    "equipment_id",
    "equipment_type",
    "total_error_count",
    "good_reading_count",
    "error_rate_percent",
    "recommended_action"
).show(10, truncate=False)

print("\n" + "=" * 90)

## Summary Report

In [0]:
print("=" * 90)
print("GOLD LAYER: EQUIPMENT ANOMALY SUMMARY - COMPLETE!")
print("=" * 90)

# Count by severity
critical_count = anomaly_verify.filter(col("severity_level") == "CRITICAL").count()
high_count = anomaly_verify.filter(col("severity_level") == "HIGH").count()
medium_count = anomaly_verify.filter(col("severity_level") == "MEDIUM").count()
normal_count = anomaly_verify.filter(col("severity_level") == "NORMAL").count()

print(f"""
    WHY WE BUILD
    Equipment anomaly summary showing sensor health by equipment_anomaly_summary.ipynb

    TABLE STRUCTURE:
    One row = One equipment
    Shows: Error count, error rate, severity, recommended action

    KEY METRICS:
    - total_error_count: How many bad readings
    - good_reading_count: How many good readings
    - error_rate_percent: % bad readings (0-100%) 
    - severity_level: CRITICAL/HIGH/MEDIUM/NORMAL
    - recommended_action: What to do about it

    SEVERITY BREAKDOWN:
    CRITICAL: {critical_count} equipment (>10% error rate - REPLACE SENSOR)
    HIGH: {high_count} equipment (5-10% error rate - INSPECT)
    MEDIUM: {medium_count} equipment(2-5% error rate - MONITOR)
    NORMAL: {normal_count} equipment (<2% error rate - OK)

    USE CASES:
    1. Sensor Maintenance: "Which sensors need replacement?"
    2. Prioritization: "What's our maintenance order?
    3. Quality Reporting: "Overall sensor health?"
    4. Root Cause Analysis: "Why is EQ-0042 failing?"
    5. Budget Planning: "How many sensors to budget for?"

    HOW MANAGERS USE IT
    Filter for CRITICAL -> Replace sensors immediatly
    Filter for HIGH -> Schedule inspection this week
    Filter for MEDIUM -> Add to next maintenance cycle
    Filter for NORMAL -> Continue normal operation

    EXAMPLE QUESTION:
    "Why does EQ-0042 have 12.5% error rate?
    Answer: "Temperature sensor is failing - recommend replacement"

    "Which factory has the worst sensor issues?"
    Answer: "Factory_South has avg 4.2% error rate (highest)
    """)

print("=" * 90)
print("GOLD LAYER: FULLY COMPLETE!")
print("1. Equipment Health Dashboard (current status)")
print("2. Daily Sensor Merics (historical trends)")
print("3. Equipment Anomaly Summary (sensor problems)")
print("=" * 90)